In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pip install piq


Perceptual / No-Reference Quality Metrics

In [ ]:
import os
from skimage import img_as_float
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import piq
import torch
from torchvision import transforms
from PIL import Image
import numpy as np

# ============================
# DEVICE
# ============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================
# TRANSFORM FOR PIQ
# ============================
to_tensor = transforms.ToTensor()

# ============================
# FUNCTION TO COMPUTE METRICS
# ============================
def image_quality_metrics(image1_path, image2_path):
    try:
        # Open the images
        img1 = Image.open(image1_path).convert("RGB")
        img2 = Image.open(image2_path).convert("RGB")

        # Convert to numpy arrays
        img1_np = np.array(img1)
        img2_np = np.array(img2)

        # Convert to float
        img1_float = img_as_float(img1_np)
        img2_float = img_as_float(img2_np)

        # SSIM (comparison between two different images)
        ssim_val = ssim(img1_float, img2_float, multichannel=True, win_size=3, data_range=1.0)

        # MS-SSIM
        img1_tensor = to_tensor(img1).unsqueeze(0)  # 1,C,H,W
        img2_tensor = to_tensor(img2).unsqueeze(0)  # 1,C,H,W
        ms_ssim_val = piq.multi_scale_ssim(img1_tensor, img2_tensor, data_range=1.0).item()

        # PSNR
        psnr_val = psnr(img1_float, img2_float, data_range=1.0)

        # BRISQUE
        brisque_val1 = piq.brisque(img1_tensor, data_range=1.0).item()
        brisque_val2 = piq.brisque(img2_tensor, data_range=1.0).item()

        return ssim_val, ms_ssim_val, psnr_val, brisque_val1, brisque_val2
    except Exception as e:
        print(f"Error processing {image1_path} and {image2_path}: {e}")
        return np.nan, np.nan, np.nan, np.nan, np.nan

# ============================
# MAIN CLASS-WISE PROCESSING
# ============================
DATA_DIR = "/content/drive/MyDrive/Combined/Original_Combined_Litchi_Jackfruit"  # Root directory containing class folders

# Loop over each class folder
for class_name in os.listdir(DATA_DIR):
    class_path = os.path.join(DATA_DIR, class_name)
    if not os.path.isdir(class_path):
        continue  # Skip files, only folders

    print(f"\n--- Processing class: {class_name} ---\n")
    image_files = [f for f in os.listdir(class_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    # Ensure there are at least two images to compare
    if len(image_files) < 2:
        print(f"Skipping class {class_name} because it has less than two images.")
        continue

    # Take the first two images in the folder (or any two images you prefer)
    img1_path = os.path.join(class_path, image_files[0])
    img2_path = os.path.join(class_path, image_files[1])

    # Calculate metrics between these two images
    ssim_val, ms_ssim_val, psnr_val, brisque_val1, brisque_val2 = image_quality_metrics(img1_path, img2_path)

    print(f"Image 1: {image_files[0]}")
    print(f"Image 2: {image_files[1]}")
    print(f"  SSIM    : {ssim_val:.4f}")
    print(f"  MS-SSIM : {ms_ssim_val:.4f}")
    print(f"  PSNR    : {psnr_val:.4f}")
    print(f"  BRISQUE (Image 1) : {brisque_val1:.4f}")
    print(f"  BRISQUE (Image 2) : {brisque_val2:.4f}\n")



--- Processing class: Mayetiola PostRain ---

Image 1: compressed_2023_08_20_13_55_IMG_2792.JPG
Image 2: compressed_2023_08_20_13_55_IMG_2790.JPG
  SSIM    : 0.8573
  MS-SSIM : 0.7692
  PSNR    : 18.3997
  BRISQUE (Image 1) : 41.4894
  BRISQUE (Image 2) : 25.1325


--- Processing class: Algal Spot Indirect ---

Image 1: compressed_IMG_1125.JPG
Image 2: compressed_IMG_1122.JPG
  SSIM    : 0.7998
  MS-SSIM : 0.7992
  PSNR    : 12.5295
  BRISQUE (Image 1) : 46.4123
  BRISQUE (Image 2) : 48.1989


--- Processing class: Leaf Mites Direct ---

Image 1: compressed_2023_08_20_14_36_IMG_3493.JPG
Image 2: compressed_2023_08_20_14_36_IMG_3488.JPG
  SSIM    : 0.7813
  MS-SSIM : 0.7832
  PSNR    : 13.4910
  BRISQUE (Image 1) : 41.5412
  BRISQUE (Image 2) : 36.9348


--- Processing class: Mature Jackfruit ---

Image 1: compressed_IMG_20250509_133515.jpg
Image 2: compressed_IMG_20250509_133441.jpg
  SSIM    : 0.8847
  MS-SSIM : 0.5773
  PSNR    : 18.0523
  BRISQUE (Image 1) : 32.8063
  BRISQUE (Imag

Blur / Sharpness Detection

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image

# ============================
# BLUR METRICS FUNCTION
# ============================
def blur_metrics(image_path):
    # Load image in grayscale
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Cannot read image: {image_path}")

    # --- Method 1: Laplacian variance ---
    lap_var = cv2.Laplacian(img, cv2.CV_64F).var()

    # --- Method 2: Tenengrad gradient ---
    gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    tenengrad = np.mean(np.sqrt(gx**2 + gy**2))

    # --- Method 3: FFT frequency domain ---
    f = np.fft.fft2(img)
    fshift = np.fft.fftshift(f)
    magnitude_spectrum = np.abs(fshift)
    total_energy = np.sum(magnitude_spectrum)
    rows, cols = img.shape
    crow, ccol = rows//2, cols//2
    mask_radius = int(0.1 * min(crow, ccol))
    low_freq_mask = np.zeros_like(img, dtype=np.uint8)
    cv2.circle(low_freq_mask, (ccol, crow), mask_radius, 1, -1)
    low_freq_energy = np.sum(magnitude_spectrum * low_freq_mask)
    high_freq_ratio = (total_energy - low_freq_energy) / total_energy

    # Return numerical values
    return {
        "Laplacian Variance": lap_var,
        "Tenengrad": tenengrad,
        "FFT High Frequency Ratio": high_freq_ratio
    }

# ============================
# MAIN CLASS-WISE LOOP
# ============================
DATA_DIR = "/content/drive/MyDrive/Combined/Original_Combined_Litchi_Jackfruit"  # Root directory containing class folders

for class_name in os.listdir(DATA_DIR):
    class_path = os.path.join(DATA_DIR, class_name)
    if not os.path.isdir(class_path):
        continue  # Skip non-folder files

    print(f"\n--- Processing class: {class_name} ---\n")
    image_files = [f for f in os.listdir(class_path) if f.lower().endswith((".jpg",".jpeg",".png"))]

    for img_file in image_files:
        img_path = os.path.join(class_path, img_file)
        try:
            metrics = blur_metrics(img_path)
            print(f"Image: {img_file}")
            for k, v in metrics.items():
                print(f"  {k}: {v:.4f}")
            print()
        except Exception as e:
            print(f"Error processing {img_file}: {e}\n")



--- Processing class: Mayetiola PostRain ---

Image: compressed_2023_08_20_13_55_IMG_2792.JPG
  Laplacian Variance: 47.9566
  Tenengrad: 11.9114
  FFT High Frequency Ratio: 0.7977


--- Processing class: Algal Spot Indirect ---

Image: compressed_IMG_1125.JPG
  Laplacian Variance: 23.0715
  Tenengrad: 10.5324
  FFT High Frequency Ratio: 0.7710


--- Processing class: Leaf Mites Direct ---

Image: compressed_2023_08_20_14_36_IMG_3493.JPG
  Laplacian Variance: 34.2466
  Tenengrad: 12.2975
  FFT High Frequency Ratio: 0.7609


--- Processing class: Mature Jackfruit ---

Image: compressed_IMG_20250509_133515.jpg
  Laplacian Variance: 200.1181
  Tenengrad: 34.2002
  FFT High Frequency Ratio: 0.8517


--- Processing class: Young Jackfruit ---

Image: compressed_Healthy_8.jpg
  Laplacian Variance: 86.8460
  Tenengrad: 23.9397
  FFT High Frequency Ratio: 0.8391


--- Processing class: Dry Leaves ---

Image: compressed_2023_08_20_14_57_IMG_4369.JPG
  Laplacian Variance: 18.4073
  Tenengrad: 8.0

3. Exposure / Brightness / Contrast Analysis

In [ ]:
import os
import cv2
import numpy as np

# ============================
# EXPOSURE / BRIGHTNESS FUNCTION
# ============================
def exposure_brightness_analysis(image_path):
    # Load image in grayscale
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Cannot read image: {image_path}")

    # --- Basic stats ---
    mean_intensity = img.mean()
    median_intensity = np.median(img)
    percentile_10 = np.percentile(img, 10)
    percentile_90 = np.percentile(img, 90)

    # --- Exposure check ---
    if mean_intensity > 200:
        exposure_status = "Overexposed"
    elif mean_intensity < 50:
        exposure_status = "Underexposed"
    else:
        exposure_status = "Normal"

    # --- Contrast analysis ---
    contrast = img.std()

    # --- Adaptive Histogram Equalization (CLAHE) ---
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_clahe = clahe.apply(img)
    mean_clahe = img_clahe.mean()
    std_clahe = img_clahe.std()

    return {
        "Mean Intensity": mean_intensity,
        "Median Intensity": median_intensity,
        "10th Percentile": percentile_10,
        "90th Percentile": percentile_90,
        "Contrast (Std Dev)": contrast,
        "CLAHE Mean": mean_clahe,
        "CLAHE Std Dev": std_clahe,
        "Exposure Status": exposure_status
    }

# ============================
# CLASS-WISE LOOP
# ============================
DATA_DIR = "/content/drive/MyDrive/Combined/Original_Combined_Litchi_Jackfruit"  # Root directory containing class folders

for class_name in os.listdir(DATA_DIR):
    class_path = os.path.join(DATA_DIR, class_name)
    if not os.path.isdir(class_path):
        continue

    print(f"\n--- Processing class: {class_name} ---\n")
    image_files = [f for f in os.listdir(class_path) if f.lower().endswith((".jpg",".jpeg",".png"))]

    for img_file in image_files:
        img_path = os.path.join(class_path, img_file)
        try:
            metrics = exposure_brightness_analysis(img_path)
            print(f"Image: {img_file}")
            for k, v in metrics.items():
                if isinstance(v, float):
                    print(f"  {k}: {v:.4f}")
                else:
                    print(f"  {k}: {v}")
            print()
        except Exception as e:
            print(f"Error processing {img_file}: {e}\n")



--- Processing class: Mayetiola PostRain ---

Image: compressed_2023_08_20_13_55_IMG_2792.JPG
  Mean Intensity: 157.9108
  Median Intensity: 188.0000
  10th Percentile: 41.0000
  90th Percentile: 206.0000
  Contrast (Std Dev): 61.4237
  CLAHE Mean: 160.1806
  CLAHE Std Dev: 53.3877
  Exposure Status: Normal


--- Processing class: Algal Spot Indirect ---

Image: compressed_IMG_1125.JPG
  Mean Intensity: 171.0954
  Median Intensity: 192.0000
  10th Percentile: 76.0000
  90th Percentile: 207.0000
  Contrast (Std Dev): 47.9779
  CLAHE Mean: 168.4884
  CLAHE Std Dev: 44.8818
  Exposure Status: Normal


--- Processing class: Leaf Mites Direct ---

Image: compressed_2023_08_20_14_36_IMG_3493.JPG
  Mean Intensity: 161.2775
  Median Intensity: 190.0000
  10th Percentile: 47.0000
  90th Percentile: 204.0000
  Contrast (Std Dev): 56.1279
  CLAHE Mean: 160.5003
  CLAHE Std Dev: 52.8052
  Exposure Status: Normal


--- Processing class: Mature Jackfruit ---

Image: compressed_IMG_20250509_133515.j

4. Color Consistency / Channel Analysis

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image

# ============================
# COLOR CONSISTENCY / CHANNEL ANALYSIS
# ============================
def color_consistency_analysis(image_path):
    # Load image in RGB
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Cannot read image: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # --- Mean per channel ---
    mean_r = img[:,:,0].mean()
    mean_g = img[:,:,1].mean()
    mean_b = img[:,:,2].mean()

    # --- Convert to HSV for saturation / hue checks ---
    img_hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    mean_hue = img_hsv[:,:,0].mean()
    mean_saturation = img_hsv[:,:,1].mean()
    mean_value = img_hsv[:,:,2].mean()

    # --- Colorfulness metric (Hasler & Suesstrunk) ---
    rg = np.abs(img[:,:,0] - img[:,:,1])
    yb = np.abs(0.5*(img[:,:,0] + img[:,:,1]) - img[:,:,2])
    std_rg = np.std(rg)
    std_yb = np.std(yb)
    mean_rg = np.mean(rg)
    mean_yb = np.mean(yb)
    colorfulness = np.sqrt(std_rg**2 + std_yb**2) + 0.3 * np.sqrt(mean_rg**2 + mean_yb**2)

    # --- Color histogram anomaly detection ---
    var_r, var_g, var_b = np.var(img[:,:,0]), np.var(img[:,:,1]), np.var(img[:,:,2])
    histogram_status = "Low contrast / uniform color" if max(var_r,var_g,var_b)<500 else "Normal"

    # Return metrics
    return {
        "Mean_R": mean_r,
        "Mean_G": mean_g,
        "Mean_B": mean_b,
        "Mean_Hue": mean_hue,
        "Mean_Saturation": mean_saturation,
        "Mean_Value": mean_value,
        "Colorfulness": colorfulness,
        "Histogram_Status": histogram_status
    }

# ============================
# CLASS-WISE LOOP
# ============================
DATA_DIR = "/content/drive/MyDrive/Combined/Original_Combined_Litchi_Jackfruit"  # Root directory containing class folders

for class_name in os.listdir(DATA_DIR):
    class_path = os.path.join(DATA_DIR, class_name)
    if not os.path.isdir(class_path):
        continue

    print(f"\n--- Processing class: {class_name} ---\n")
    image_files = [f for f in os.listdir(class_path) if f.lower().endswith((".jpg",".jpeg",".png"))]

    for img_file in image_files:
        img_path = os.path.join(class_path, img_file)
        try:
            metrics = color_consistency_analysis(img_path)
            print(f"Image: {img_file}")
            for k, v in metrics.items():
                if isinstance(v,float):
                    print(f"  {k}: {v:.4f}")
                else:
                    print(f"  {k}: {v}")
            print()
        except Exception as e:
            print(f"Error processing {img_file}: {e}\n")



--- Processing class: Mayetiola PostRain ---

Image: compressed_2023_08_20_13_55_IMG_2792.JPG
  Mean_R: 159.7949
  Mean_G: 158.7271
  Mean_B: 148.6584
  Mean_Hue: 22.6722
  Mean_Saturation: 43.6421
  Mean_Value: 161.2796
  Colorfulness: 156.3932
  Histogram_Status: Normal


--- Processing class: Algal Spot Indirect ---

Image: compressed_IMG_1125.JPG
  Mean_R: 172.6848
  Mean_G: 173.1306
  Mean_B: 156.8477
  Mean_Hue: 25.7355
  Mean_Saturation: 42.8380
  Mean_Value: 175.5898
  Colorfulness: 161.5720
  Histogram_Status: Normal


--- Processing class: Leaf Mites Direct ---

Image: compressed_2023_08_20_14_36_IMG_3493.JPG
  Mean_R: 161.9394
  Mean_G: 162.9363
  Mean_B: 151.1821
  Mean_Hue: 28.2184
  Mean_Saturation: 42.2061
  Mean_Value: 164.2714
  Colorfulness: 182.4106
  Histogram_Status: Normal


--- Processing class: Mature Jackfruit ---

Image: compressed_IMG_20250509_133515.jpg
  Mean_R: 169.8684
  Mean_G: 149.5833
  Mean_B: 136.9327
  Mean_Hue: 21.6021
  Mean_Saturation: 64.6643
 

6. Noise Estimation

In [ ]:
import os
import cv2
import numpy as np
from skimage.restoration import estimate_sigma
from PIL import Image

# ============================
# NOISE ESTIMATION FUNCTION
# ============================
def noise_estimation(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Cannot read image: {image_path}")

    # --- Gaussian noise estimation using skimage ---
    sigma_est = estimate_sigma(img)

    # --- Salt-and-pepper detection ---
    sp_ratio = np.sum((img==0) | (img==255)) / img.size

    # --- Local variance (Laplacian) ---
    lap_var = cv2.Laplacian(img, cv2.CV_64F).var()

    return {
        "Estimated Gaussian Noise Sigma": sigma_est,
        "Salt-Pepper Ratio": sp_ratio,
        "Local Variance (Laplacian)": lap_var
    }

# ============================
# CLASS-WISE LOOP
# ============================
DATA_DIR = "/content/drive/MyDrive/Combined/Original_Combined_Litchi_Jackfruit"  # Root directory containing class folders

for class_name in os.listdir(DATA_DIR):
    class_path = os.path.join(DATA_DIR, class_name)
    if not os.path.isdir(class_path):
        continue

    print(f"\n--- Processing class: {class_name} ---\n")
    image_files = [f for f in os.listdir(class_path) if f.lower().endswith((".jpg",".jpeg",".png"))]

    for img_file in image_files:
        img_path = os.path.join(class_path, img_file)
        try:
            metrics = noise_estimation(img_path)
            print(f"Image: {img_file}")
            for k, v in metrics.items():
                print(f"  {k}: {v:.4f}")
            print()
        except Exception as e:
            print(f"Error processing {img_file}: {e}\n")



--- Processing class: Mayetiola PostRain ---

Image: compressed_2023_08_20_13_55_IMG_2792.JPG
  Estimated Gaussian Noise Sigma: 0.0000
  Salt-Pepper Ratio: 0.0001
  Local Variance (Laplacian): 47.9566


--- Processing class: Algal Spot Indirect ---

Image: compressed_IMG_1125.JPG
  Estimated Gaussian Noise Sigma: 0.0745
  Salt-Pepper Ratio: 0.0000
  Local Variance (Laplacian): 23.0715


--- Processing class: Leaf Mites Direct ---

Image: compressed_2023_08_20_14_36_IMG_3493.JPG
  Estimated Gaussian Noise Sigma: 0.1853
  Salt-Pepper Ratio: 0.0000
  Local Variance (Laplacian): 34.2466


--- Processing class: Mature Jackfruit ---

Image: compressed_IMG_20250509_133515.jpg
  Estimated Gaussian Noise Sigma: 0.2350
  Salt-Pepper Ratio: 0.0000
  Local Variance (Laplacian): 200.1181


--- Processing class: Young Jackfruit ---

Image: compressed_Healthy_8.jpg
  Estimated Gaussian Noise Sigma: 0.0000
  Salt-Pepper Ratio: 0.0000
  Local Variance (Laplacian): 86.8460


--- Processing class: Dry L

7. Duplicate / Near-Duplicate Detection

In [ ]:
!pip install ImageHash


In [ ]:
import os
from PIL import Image
import imagehash

# ============================
# DUPLICATE / NEAR-DUPLICATE DETECTION
# ============================
def find_duplicates(image_dir):
    valid_formats = (".png", ".jpg", ".jpeg")
    hashes = {}
    duplicates = []

    for root, _, files in os.walk(image_dir):
        for img_file in files:
            if not img_file.lower().endswith(valid_formats):
                continue
            img_path = os.path.join(root, img_file)
            try:
                img = Image.open(img_path).convert("RGB")
                h = imagehash.phash(img)
                if h in hashes:
                    duplicates.append((hashes[h], img_path))
                else:
                    hashes[h] = img_path
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
                continue
    return duplicates

# ============================
# CLASS-WISE LOOP
# ============================
DATA_DIR = "/content/drive/MyDrive/Combined/Original_Combined_Litchi_Jackfruit"  # Root directory containing class folders

for class_name in os.listdir(DATA_DIR):
    class_path = os.path.join(DATA_DIR, class_name)
    if not os.path.isdir(class_path):
        continue

    print(f"\n--- Processing class: {class_name} ---\n")
    duplicates = find_duplicates(class_path)
    if not duplicates:
        print("No duplicates found.\n")
    else:
        print("Duplicate / Near-Duplicate Pairs:")
        for pair in duplicates:
            print(f"  {pair[0]}  <-->  {pair[1]}")
        print()



--- Processing class: Mayetiola PostRain ---

No duplicates found.


--- Processing class: Algal Spot Indirect ---

No duplicates found.


--- Processing class: Leaf Mites Direct ---

No duplicates found.


--- Processing class: Mature Jackfruit ---

No duplicates found.


--- Processing class: Young Jackfruit ---

No duplicates found.


--- Processing class: Dry Leaves ---

No duplicates found.


--- Processing class: Anthracnose Cloudy ---

No duplicates found.


--- Processing class: Entomosporium Spot ---

No duplicates found.



Cell 8: Deep Learning / Embedding Based Quality Checks

In [ ]:
import torch
from torchvision import models, transforms
from PIL import Image
import numpy as np
from sklearn.neighbors import NearestNeighbors
import os

def deep_learning_outlier_detection(image_dir):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = models.resnet18(pretrained=True)
    model = model.to(device)
    model.eval()
    feature_extractor = torch.nn.Sequential(*list(model.children())[:-1])

    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])

    all_features = []
    all_image_files = []
    batch_size = 32 # Define a batch size

    with torch.no_grad():
        current_batch = []
        current_batch_files = []
        for root, _, files in os.walk(image_dir): # Use os.walk to traverse subdirectories
            for img_file in files:
                if not img_file.lower().endswith((".png",".jpg",".jpeg")):
                    continue
                img_path = os.path.join(root, img_file) # Construct the full path
                try:
                    img = Image.open(img_path).convert("RGB")
                    img_tensor = transform(img)
                    current_batch.append(img_tensor)
                    current_batch_files.append(img_path)

                    if len(current_batch) == batch_size:
                        batch_tensor = torch.stack(current_batch).to(device)
                        feat = feature_extractor(batch_tensor).view(len(current_batch), -1).cpu().detach().numpy()
                        all_features.append(feat)
                        all_image_files.extend(current_batch_files)
                        current_batch = []
                        current_batch_files = []

                except Exception as e:
                     print(f"Error processing {img_path}: {e}") # Add error printing
                     continue

    # Process any remaining images in the last batch
    if current_batch:
        batch_tensor = torch.stack(current_batch).to(device)
        feat = feature_extractor(batch_tensor).view(len(current_batch), -1).cpu().detach().numpy()
        all_features.append(feat)
        all_image_files.extend(current_batch_files)

    features = np.concatenate(all_features, axis=0) if all_features else np.array([])
    image_files = all_image_files


    if len(features) == 0:
        print("No features extracted. Please check image directory and file formats.")
        return [] # Return an empty list if no features extracted
    elif len(features) < 5: # Add check for minimum number of samples for NearestNeighbors
         print(f"Not enough samples ({len(features)}) to perform outlier detection with n_neighbors=5. Skipping outlier detection.")
         return [] # Return empty list if not enough samples
    else:
        # Ensure n_neighbors is not more than the number of samples
        n_neighbors = min(5, len(features))
        nbrs = NearestNeighbors(n_neighbors=n_neighbors, metric='euclidean').fit(features)
        distances, indices = nbrs.kneighbors(features)
        mean_dist = distances.mean(axis=1)
        threshold = np.percentile(mean_dist, 95)
        outliers = [image_files[i] for i,d in enumerate(mean_dist) if d > threshold]

        print("Potential visual outlier images:")
        for f in outliers:
            print(f"- {f}")

        return outliers # Return the list of outliers

# Example
image_dir = "/content/drive/MyDrive/Combined/Original_Combined_Litchi_Jackfruit" # Path to your image directory
outliers = deep_learning_outlier_detection(image_dir)
print("Potential outlier images based on embeddings:")
for f in outliers:
    print(f"- {f}")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Potential visual outlier images:
- /content/drive/MyDrive/Combined/Original_Combined_Litchi_Jackfruit/Mature Jackfruit/compressed_IMG_20250509_133515.jpg
Potential outlier images based on embeddings:
- /content/drive/MyDrive/Combined/Original_Combined_Litchi_Jackfruit/Mature Jackfruit/compressed_IMG_20250509_133515.jpg


Cell 9: Compression / Artifact Detection

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image

# ============================
# COMPRESSION / ARTIFACT DETECTION
# ============================
def compression_artifact_detection(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Cannot read image: {image_path}")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # --- JPEG blocking artifact detection ---
    block_size = 8
    h, w = gray.shape
    blocks_vertical = gray[:h//block_size*block_size, :w//block_size*block_size].reshape(h//block_size, block_size, -1, block_size)
    vertical_diff = np.mean(np.abs(blocks_vertical[:, :-1, :, :] - blocks_vertical[:, 1:, :, :]))
    horizontal_diff = np.mean(np.abs(blocks_vertical[:, :, :-1, :] - blocks_vertical[:, :, 1:, :]))
    block_artifact_score = vertical_diff + horizontal_diff

    # --- High-frequency / ringing approximation ---
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()

    return {
        "Block Artifact Score": block_artifact_score,
        "High Frequency Variance": lap_var
    }

# ============================
# CLASS-WISE LOOP
# ============================
DATA_DIR = "/content/drive/MyDrive/Combined/Original_Combined_Litchi_Jackfruit"  # Root directory containing class folders

for class_name in os.listdir(DATA_DIR):
    class_path = os.path.join(DATA_DIR, class_name)
    if not os.path.isdir(class_path):
        continue

    print(f"\n--- Processing class: {class_name} ---\n")
    image_files = [f for f in os.listdir(class_path) if f.lower().endswith((".jpg",".jpeg",".png"))]

    for img_file in image_files:
        img_path = os.path.join(class_path, img_file)
        try:
            metrics = compression_artifact_detection(img_path)
            print(f"Image: {img_file}")
            for k,v in metrics.items():
                print(f"  {k}: {v:.4f}")
            print()
        except Exception as e:
            print(f"Error processing {img_file}: {e}\n")



--- Processing class: Mayetiola PostRain ---

Image: compressed_2023_08_20_13_55_IMG_2792.JPG
  Block Artifact Score: 127.0627
  High Frequency Variance: 47.5952


--- Processing class: Algal Spot Indirect ---

Image: compressed_IMG_1125.JPG
  Block Artifact Score: 154.8736
  High Frequency Variance: 23.0466


--- Processing class: Leaf Mites Direct ---

Image: compressed_2023_08_20_14_36_IMG_3493.JPG
  Block Artifact Score: 163.1852
  High Frequency Variance: 33.9683


--- Processing class: Mature Jackfruit ---

Image: compressed_IMG_20250509_133515.jpg
  Block Artifact Score: 169.1916
  High Frequency Variance: 199.9574


--- Processing class: Young Jackfruit ---

Image: compressed_Healthy_8.jpg
  Block Artifact Score: 136.0536
  High Frequency Variance: 85.4648


--- Processing class: Dry Leaves ---

Image: compressed_2023_08_20_14_57_IMG_4369.JPG
  Block Artifact Score: 109.0518
  High Frequency Variance: 18.2851


--- Processing class: Anthracnose Cloudy ---

Image: compressed_20

Fourier / Frequency Domain Analysis

In [ ]:
import os
import cv2
import numpy as np
import pywt  # Wavelet transform
from PIL import Image

# ============================
# FOURIER / FREQUENCY DOMAIN ANALYSIS
# ============================
def frequency_domain_analysis(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Cannot read image: {image_path}")

    # --- FFT / High-frequency vs Low-frequency energy ---
    f = np.fft.fft2(img)
    fshift = np.fft.fftshift(f)
    magnitude = np.abs(fshift)
    total_energy = np.sum(magnitude)

    # Create low-frequency mask (central 20% area)
    h, w = img.shape
    crow, ccol = h//2, w//2
    mask_radius_h = int(0.1*h)
    mask_radius_w = int(0.1*w)
    low_freq_mask = np.zeros_like(img, dtype=np.uint8)
    low_freq_mask[crow-mask_radius_h:crow+mask_radius_h, ccol-mask_radius_w:ccol+mask_radius_w] = 1
    low_energy = np.sum(magnitude * low_freq_mask)
    high_energy_ratio = (total_energy - low_energy) / total_energy

    # --- Wavelet Transform for localized frequency anomalies ---
    coeffs = pywt.dwt2(img, 'haar')
    cA, (cH, cV, cD) = coeffs
    wavelet_energy = np.mean(np.abs(cH)) + np.mean(np.abs(cV)) + np.mean(np.abs(cD))

    return {
        "High-Frequency Energy Ratio": high_energy_ratio,
        "Wavelet Detail Energy": wavelet_energy
    }

# ============================
# CLASS-WISE LOOP
# ============================
DATA_DIR = "/content/drive/MyDrive/Combined/Original_Combined_Litchi_Jackfruit"  # Root directory containing class folders

for class_name in os.listdir(DATA_DIR):
    class_path = os.path.join(DATA_DIR, class_name)
    if not os.path.isdir(class_path):
        continue

    print(f"\n--- Processing class: {class_name} ---\n")
    image_files = [f for f in os.listdir(class_path) if f.lower().endswith((".jpg",".jpeg",".png"))]

    for img_file in image_files:
        img_path = os.path.join(class_path, img_file)
        try:
            metrics = frequency_domain_analysis(img_path)
            print(f"Image: {img_file}")
            for k, v in metrics.items():
                print(f"  {k}: {v:.6f}")
            print()
        except Exception as e:
            print(f"Error processing {img_file}: {e}\n")



--- Processing class: Mayetiola PostRain ---

Image: compressed_2023_08_20_13_55_IMG_2792.JPG
  High-Frequency Energy Ratio: 0.633878
  Wavelet Detail Energy: 2.446426


--- Processing class: Algal Spot Indirect ---

Image: compressed_IMG_1125.JPG
  High-Frequency Energy Ratio: 0.609139
  Wavelet Detail Energy: 1.805083


--- Processing class: Leaf Mites Direct ---

Image: compressed_2023_08_20_14_36_IMG_3493.JPG
  High-Frequency Energy Ratio: 0.643862
  Wavelet Detail Energy: 2.493635


--- Processing class: Mature Jackfruit ---

Image: compressed_IMG_20250509_133515.jpg
  High-Frequency Energy Ratio: 0.675027
  Wavelet Detail Energy: 7.593171


--- Processing class: Young Jackfruit ---

Image: compressed_Healthy_8.jpg
  High-Frequency Energy Ratio: 0.663584
  Wavelet Detail Energy: 4.811882


--- Processing class: Dry Leaves ---

Image: compressed_2023_08_20_14_57_IMG_4369.JPG
  High-Frequency Energy Ratio: 0.554698
  Wavelet Detail Energy: 1.345150


--- Processing class: Anthracno